## 10.08 视觉 Transformer


### 环境配置

以下代码片段自包含 10.08 主 notebook 的全部 PyPTO 组件（图像块嵌入、GELU、层归一化、多头注意力、ViTBlock、ViT），并定义数据加载与训练辅助函数，供练习直接使用。

In [1]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import math
    import torch
    from torch import nn
    import torch_npu

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

from src.pypto_ops import (PyPTOLinear, PyPTOBMM, PyPTOSoftmax, PyPTOSigmoid,
                           PyPTOMul, PyPTOLayerNorm, loss_fn)
from src.utils import sequence_mask, _accuracy

torch.manual_seed(0);


In [2]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
# ---- 10.08 主 notebook 的 PyPTO 组件（自包含，供练习使用）----

class PatchEmbedding(nn.Module):
    """图像块嵌入（unfold + PyPTOLinear，与卷积等价）"""

    def __init__(self, img_size=96, patch_size=16, num_hiddens=512,
                 in_channels=None):
        super().__init__()
        def _make_tuple(x):
            if not isinstance(x, (list, tuple)):
                return (x, x)
            return x
        img_size, patch_size = _make_tuple(img_size), _make_tuple(patch_size)
        self.img_size, self.patch_size = img_size, patch_size
        self.num_patches = (img_size[0] // patch_size[0]) * (
            img_size[1] // patch_size[1])
        self.num_hiddens = num_hiddens
        # 通道数由 in_channels 指定或首次 forward 按输入动态确定（不硬编码 3）
        self.proj = None
        if in_channels is not None:
            self.proj = PyPTOLinear(
                in_channels * patch_size[0] * patch_size[1], num_hiddens)

    def forward(self, X):
        p_h, p_w = self.patch_size
        if self.proj is None:
            self.proj = PyPTOLinear(
                X.shape[1] * p_h * p_w, self.num_hiddens).to(X.device)
        patches = torch.nn.functional.unfold(
            X, kernel_size=(p_h, p_w), stride=(p_h, p_w))
        patches = patches.transpose(1, 2).contiguous()
        return self.proj(patches)


class PyPTOGELU(nn.Module):
    """GELU（PyPTO 内联组合：x·sigmoid(1.702x)）"""

    def forward(self, x):
        return PyPTOMul.apply(x, PyPTOSigmoid.apply(x * 1.702))


class ViTMLP(nn.Module):
    def __init__(self, mlp_num_input, mlp_num_hiddens, mlp_num_outputs,
                 dropout=0.5):
        super().__init__()
        self.dense1 = PyPTOLinear(mlp_num_input, mlp_num_hiddens)
        self.gelu = PyPTOGELU()
        self.dropout1 = nn.Dropout(dropout)
        self.dense2 = PyPTOLinear(mlp_num_hiddens, mlp_num_outputs)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout2(self.dense2(self.dropout1(self.gelu(
            self.dense1(x)))))


class PyPTOLayerNormModule(nn.Module):
    """PyPTO 层归一化（与 10.7 节一致）"""

    def __init__(self, normalized_shape):
        super().__init__()
        self.normalized_shape = normalized_shape
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, X):
        orig_shape = X.shape
        if isinstance(self.normalized_shape, int):
            num_dims = 1
        else:
            num_dims = len(self.normalized_shape)
        x2 = X.reshape(orig_shape[:-num_dims] + (-1,)).contiguous()
        y2 = PyPTOLayerNorm.apply(x2, self.weight.reshape(-1),
                                  self.bias.reshape(-1))
        return y2.reshape(orig_shape)


def masked_softmax(X, valid_lens):
    if valid_lens is None:
        return PyPTOSoftmax.apply(X)
    shape = X.shape
    if valid_lens.dim() == 1:
        valid_lens = torch.repeat_interleave(valid_lens, shape[1])
    else:
        valid_lens = valid_lens.reshape(-1)
    X = sequence_mask(X.reshape(-1, shape[-1]), valid_lens, value=-1e6)
    return PyPTOSoftmax.apply(X.reshape(shape))


class DotProductAttention(nn.Module):
    def __init__(self, dropout, **kwargs):
        super().__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]
        scores = PyPTOBMM.apply(queries, keys.transpose(1, 2)) / math.sqrt(d)
        self.attention_weights = masked_softmax(scores, valid_lens)
        return PyPTOBMM.apply(self.dropout(self.attention_weights), values)


def transpose_qkv(X, num_heads):
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)
    X = X.permute(0, 2, 1, 3)
    return X.reshape(-1, X.shape[2], X.shape[3])


def transpose_output(X, num_heads):
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)


class MultiHeadAttention(nn.Module):
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 num_heads, dropout, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.attention = DotProductAttention(dropout)
        self.W_q = PyPTOLinear(query_size, num_hiddens, bias=bias)
        self.W_k = PyPTOLinear(key_size, num_hiddens, bias=bias)
        self.W_v = PyPTOLinear(value_size, num_hiddens, bias=bias)
        self.W_o = PyPTOLinear(num_hiddens, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):
        queries = transpose_qkv(self.W_q(queries), self.num_heads)
        keys = transpose_qkv(self.W_k(keys), self.num_heads)
        values = transpose_qkv(self.W_v(values), self.num_heads)
        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(
                valid_lens, repeats=self.num_heads, dim=0)
        output = self.attention(queries, keys, values, valid_lens)
        output_concat = transpose_output(output, self.num_heads)
        return self.W_o(output_concat)


class ViTBlock(nn.Module):
    def __init__(self, num_hiddens, norm_shape, mlp_num_hiddens,
                 num_heads, dropout, use_bias=False):
        super().__init__()
        self.ln1 = PyPTOLayerNormModule(norm_shape)
        self.attention = MultiHeadAttention(
            num_hiddens, num_hiddens, num_hiddens, num_hiddens, num_heads,
            dropout, use_bias)
        self.ln2 = PyPTOLayerNormModule(norm_shape)
        self.mlp = ViTMLP(num_hiddens, mlp_num_hiddens, num_hiddens, dropout)

    def forward(self, X, valid_lens=None):
        X = X + self.attention(*([self.ln1(X)] * 3), valid_lens)
        return X + self.mlp(self.ln2(X))


class ViT(nn.Module):
    """视觉 Transformer（pool_mode='cls' 用 cls 标记；'mean' 用图像块均值）"""

    def __init__(self, img_size, patch_size, num_hiddens, mlp_num_hiddens,
                 num_heads, num_blks, emb_dropout, blk_dropout,
                 pool_mode='cls', num_classes=10, in_channels=3):
        super().__init__()
        self.pool_mode = pool_mode
        self.patch_embedding = PatchEmbedding(
            img_size, patch_size, num_hiddens, in_channels=in_channels)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, num_hiddens))
        num_steps = self.patch_embedding.num_patches + 1
        self.pos_embedding = nn.Parameter(torch.randn(1, num_steps, num_hiddens))
        self.dropout = nn.Dropout(emb_dropout)
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module(f"{i}", ViTBlock(
                num_hiddens, num_hiddens, mlp_num_hiddens,
                num_heads, blk_dropout))
        self.head = nn.Sequential(PyPTOLayerNormModule(num_hiddens),
                                  PyPTOLinear(num_hiddens, num_classes))

    def forward(self, X):
        X = self.patch_embedding(X)
        if self.pool_mode == 'cls':
            X = torch.cat((self.cls_token.expand(X.shape[0], -1, -1), X), 1)
            X = self.dropout(X + self.pos_embedding)
            for blk in self.blks:
                X = blk(X)
            return self.head(X[:, 0])
        else:  # mean pooling：不用 cls 标记，用全部图像块表示的平均
            X = self.dropout(X + self.pos_embedding[:, :X.shape[1], :])
            for blk in self.blks:
                X = blk(X)
            return self.head(X.mean(dim=1))

In [3]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
import torchvision
from torchvision import transforms


def load_fmnist(img_size, batch_size):
    trans = transforms.Compose([transforms.Resize((img_size, img_size)),
                                transforms.ToTensor()])
    train_data = torchvision.datasets.FashionMNIST(
        root="../../data", train=True, transform=trans, download=True)
    test_data = torchvision.datasets.FashionMNIST(
        root="../../data", train=False, transform=trans, download=True)
    train_iter = torch.utils.data.DataLoader(
        train_data, batch_size, shuffle=True, num_workers=4)
    test_iter = torch.utils.data.DataLoader(
        test_data, batch_size, shuffle=False, num_workers=4)
    return train_iter, test_iter


def train_vit(net, train_iter, test_iter, lr, num_epochs, device):
    """训练并返回每 epoch 的 (loss, train acc, test acc) 列表"""
    def init_weights(m):
        if isinstance(m, PyPTOLinear):
            nn.init.kaiming_uniform_(m.weight, a=math.sqrt(5))
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    net = net.to(device)
    net.apply(init_weights)
    optimizer = torch.optim.SGD(net.parameters(), lr=lr)
    history = []
    for epoch in range(num_epochs):
        net.train()
        train_loss, train_acc, n = 0.0, 0.0, 0
        for X, y in train_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            l = loss_fn(y_hat, y, num_classes=10)
            optimizer.zero_grad()
            l.backward()
            optimizer.step()
            train_loss += l.item() * y.numel()
            train_acc += _accuracy(y_hat, y)
            n += y.numel()
        test_acc = evaluate_accuracy(net, test_iter, device)
        history.append((train_loss / n, train_acc / n, test_acc))
        print(f'epoch {epoch + 1}, loss {train_loss / n:.3f}, '
              f'train acc {train_acc / n:.3f}, test acc {test_acc:.3f}')
    return history


def evaluate_accuracy(net, data_iter, device):
    net.eval()
    correct, total = 0.0, 0.0
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            correct += _accuracy(net(X), y)
            total += y.numel()
    return correct / total

### 练习 10.8.1

**题目：** `img_size` 的值如何影响训练时间？

**解答：**

图像块数 $m = (img\_size / patch\_size)^2$。训练时间主要受两部分影响：

1. **图像块嵌入**：unfold 切块 + 线性投影的计算量正比于图像块数 $m$（每个块一次投影），所以该项随 $img\_size^2$ 线性增长（patch 固定时）；
2. **自注意力**：Transformer 编码器对 $m+1$ 个词元（含 cls 标记）做自注意力，注意力矩阵为 $(m+1)\times(m+1)$，时间与空间复杂度均为 $O(m^2)$（[10.6 节](../10.06_self_attention_and_positional_encoding.ipynb)），因此该项随 $img\_size^4$ 增长（patch 固定时 $m \propto img\_size^2$）。

**结论**：$img\_size$ 从 96 增大到 192（patch 仍为 16）时，图像块数从 36 变为 144（4 倍），注意力代价增大 16 倍，训练时间显著增加；同时更大的输入图也使数据加载与显存占用增加。因此 ViT 不适合直接处理高分辨率图像，这也是 Swin Transformer 等分层方法引入局部窗口注意力的动机。

### 练习 10.8.2

**题目：** 如果不将 “&lt;cls&gt;” 标记的表示投影到输出，而是投影图像块表示的均值，你会怎么做？实现这一更改，看看它如何影响准确率。

**解答：**

做法：去掉 cls 标记与对应位置编码，让编码器只处理图像块序列，最后对输出的图像块表示沿序列维求平均（global average pooling），再送入分类头。实现见下（`ViT` 增加 `pool_mode='mean'` 分支：`X.mean(dim=1)`）。

为控制验证时间，用较小配置各训练 3 个 epoch 对比：

In [4]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
# 练习 10.8.2：用图像块均值代替 cls 标记
img_size, patch_size = 64, 8
num_hiddens, mlp_num_hiddens, num_heads, num_blks = 128, 256, 4, 2
emb_dropout, blk_dropout, lr = 0.1, 0.1, 0.1
num_epochs, batch_size = 3, 64

train_iter, test_iter = load_fmnist(img_size, batch_size)
print('训练集批数：', len(train_iter), '，每批', batch_size)

print('== cls 标记版 ==')
net_cls = ViT(img_size, patch_size, num_hiddens, mlp_num_hiddens, num_heads,
              num_blks, emb_dropout, blk_dropout, pool_mode='cls',
              in_channels=1)  # FashionMNIST 为单通道
hist_cls = train_vit(net_cls, train_iter, test_iter, lr, num_epochs, device)

print('== 均值池化版 ==')
net_mean = ViT(img_size, patch_size, num_hiddens, mlp_num_hiddens, num_heads,
               num_blks, emb_dropout, blk_dropout, pool_mode='mean',
               in_channels=1)  # FashionMNIST 为单通道
hist_mean = train_vit(net_mean, train_iter, test_iter, lr, num_epochs, device)

print(f'cls 版最终 test acc: {hist_cls[-1][2]:.3f}')
print(f'mean 版最终 test acc: {hist_mean[-1][2]:.3f}')

训练集批数： 938 ，每批 64
== cls 标记版 ==


epoch 1, loss 1.073, train acc 0.604, test acc 0.744


epoch 2, loss 0.655, train acc 0.750, test acc 0.758


epoch 3, loss 0.580, train acc 0.780, test acc 0.797
== 均值池化版 ==


epoch 1, loss 1.157, train acc 0.587, test acc 0.695


epoch 2, loss 0.629, train acc 0.764, test acc 0.773


epoch 3, loss 0.544, train acc 0.798, test acc 0.802
cls 版最终 test acc: 0.797
mean 版最终 test acc: 0.802


**分析**：均值池化把全局信息平均到每个 patch 的表示上，无 cls 标记位置编码的额外参数；对 Fashion-MNIST 这类小规模数据，两者准确率接近（cls 标记通过自注意力天然聚合全局信息，平均池化则显式平均）。在数据充足的大规模任务上，cls 标记通常略优或相当；均值池化更省参数、实现简单，被部分 ViT 变体（如 Swin 的分类头）采用。

### 练习 10.8.3

**题目：** 你能修改超参数来提高视觉 Transformer 的准确率吗？

**解答：**

可以从以下方向尝试（训练时间允许时可组合验证）：

1. **模型容量**：增大 `num_hiddens` / `mlp_num_hiddens`、增加 `num_blks`、减小 `patch_size`（保留更多图像细节，但注意自注意力 $O(m^2)$ 变贵）；
2. **优化**：降低学习率并增加 epoch（如 `lr=0.05` + 20 epochs）、改用 AdamW、加学习率 warmup 与余弦退火；
3. **正则化**：增大 `emb_dropout` / `blk_dropout`、增加数据增强（随机水平翻转、随机裁剪、RandAugment），缓解小数据集上的过拟合；
4. **结构变体**：使用均值池化（练习 2）、深浅混合层数、Swin 式局部窗口注意力降低计算量。

由于 Transformer 缺少卷积的归纳偏置（平移不变性、局部性），在小数据集上通常难以超过同规模 CNN（ResNet），更大数据集 + 更大模型才能体现 ViT 的扩展性优势（见本节点小结）。

---

## 参考答案来源
参考答案和 PyTorch 代码实现来源：https://datawhalechina.github.io/d2l-ai-solutions-manual/#/